search engine with tools and agents 

#arxiv research 
#tools creation

In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper


In [2]:
# used the inbuilt tools of wikipedia 
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
api_arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_arxiv_wrapper)
arxiv.name

'arxiv'

In [4]:
#combining the abouve tools
tools=[wiki,arxiv]

In [12]:
#custom tools - rag tool 
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq


In [6]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [8]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = FAISS.from_documents(documents, embeddings)

retriever = vectordb.as_retriever()

print("done")



done


In [9]:
# to use this as a tool 
# at end we need to have a retriever and by that we will be able to create langsmith search 
from langchain.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about langsmith")

retriever_tool.name

'langsmith-search'

In [11]:
tools=[wiki,arxiv,retriever_tool]  #list of tools 
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/apple/DEVELOPER/rag_env/lib/python3.11/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 Tool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x10d99b880>, retriever=VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x32c245510>), document

In [ ]:
# run all this tools with agents and llm models

# tool, llms --> agentexecuter 


In [16]:
import os 
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])




In [23]:
# agents 

from langchain.agents import create_openai_tools_agent, AgentExecutor

agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)


In [24]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

#run
response = agent_executor.invoke({
    "input": "what is ai?"
})

print(response)



> Entering new AgentExecutor chain...
AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.

AI technology is divided into two main categories:

1. **Narrow or Weak AI**: This type of AI is designed and trained for a specific task, such as facial recognition, language translation, or playing chess. Narrow AI is not capable of general reasoning or self-awareness.

2. **General or Strong AI**: This type of AI is designed to perform any intellectual task that a human can, such as reasoning, problem-solving, and learning. General AI is still in the theoretical stage and is considered a long-term goal for AI research.

AI systems use various techniques to learn and improve their performance, including:

1. **Machine learning**: This involves training AI models on large da

In [25]:
response = agent_executor.invoke({
    "input": "what is machine learning ?"
})

print(response)



> Entering new AgentExecutor chain...
Machine learning (ML) is a subset of artificial intelligence (AI) that involves training algorithms to learn from data, allowing them to make predictions, classify objects, or make decisions without being explicitly programmed.

Machine learning works by analyzing large datasets and identifying patterns, relationships, and trends. The algorithms used in ML are designed to improve their performance over time, through experience and data, rather than relying on explicit instructions.

There are several key aspects of machine learning:

1. **Training**: In this phase, the algorithm is fed a large dataset, and it learns to identify patterns and relationships within the data.
2. **Modeling**: Once the algorithm has learned from the data, it creates a model that can be used to make predictions or classify new data.
3. **Testing**: The model is tested on new, unseen data to evaluate its performance and accuracy.
4. **Deployment**: The trained model is d